# Piper Vietnamese (sea-g2p) - Single Speaker Training (Kaggle & Colab)
Huấn luyện mô hình 1 giọng (`num_speakers=1`, ví dụ dataset dạng `utt_00000.wav|ngochuyen|ma môn`), chất lượng `medium` (22050 Hz), warmstart từ checkpoint single-speaker Lessac medium.

In [ ]:
!nvidia-smi

In [ ]:
# 1. Cấu hình (Tự động nhận diện Kaggle hoặc Colab)
import os

IS_KAGGLE = os.path.exists("/kaggle/working")
REPO_URL  = "https://github.com/18200245/piper-seag2p.git"
VOICE     = "ngochuyen"

if IS_KAGGLE:
    # Trên Kaggle: điền đường dẫn thư mục dataset trong /kaggle/input (hoặc để trống để tự tìm)
    DATASET_PATH = ""  # Ví dụ: "/kaggle/input/datasets/xkorenguyenthinh/ngochuyen"
    WORK         = "/kaggle/working/piper_work"
    OUT          = "/kaggle/working/piper_out"
else:
    # Trên Colab: mount Drive để lưu checkpoint và chỉ định file .zip hoặc thư mục dataset
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_PATH = "/content/ngochuyen.zip"  # Hoặc thư mục chứa metadata.csv và wavs
    WORK         = "/content/work"
    OUT          = f"/content/drive/MyDrive/piper_{VOICE}"

# Base checkpoint single-speaker medium (22050 Hz, gin_channels=0 khớp 100% với 1 giọng)
BASE_REPO         = "rhasspy/piper-checkpoints"
BASE_CKPT_IN_REPO = "en/en_US/lessac/medium/epoch=2164-step=1355540.ckpt"

BATCH_SIZE   = 16   # Đặt 12-16 cho T4/L4 (16-22GB VRAM); có thể tăng lên 32-48 trên A100
MAX_EPOCHS   = 300
MIN_DURATION = 0.5
MAX_DURATION = 15.0

os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab'} | WORK={WORK} | OUT={OUT}")

In [ ]:
# 2. Cài đặt môi trường & biên dịch monotonic_align
!apt-get -qq update && apt-get -qq install -y espeak-ng build-essential > /dev/null
!rm -rf {WORK}/piper
!git clone {REPO_URL} {WORK}/piper
%cd {WORK}/piper
!pip install -q -e ".[train,vi]" huggingface_hub soundfile
!bash build_monotonic_align.sh

In [ ]:
# 3. Kiểm tra nhanh sea-g2p phonemizer
import sys
if f"{WORK}/piper/src" not in sys.path:
    sys.path.insert(0, f"{WORK}/piper/src")

from piper.phonemize_vietnamese import VietnamesePhonemizer
print(VietnamesePhonemizer().phonemize("Chương hai nghìn tám trăm mười. Ma môn."))

In [ ]:
# 4. Chuẩn bị & làm sạch CSV cho 1 giọng (hỗ trợ cả dạng utt_00000.wav|ngochuyen|ma môn và utt_00000.wav|ma môn)
import glob, os
import soundfile as sf

# Giải nén nếu DATASET_PATH là file .zip
if DATASET_PATH and DATASET_PATH.endswith(".zip") and os.path.exists(DATASET_PATH):
    data_root = os.path.join(WORK, "data")
    os.makedirs(data_root, exist_ok=True)
    !unzip -q -o "{DATASET_PATH}" -d "{data_root}"
elif DATASET_PATH and os.path.exists(DATASET_PATH):
    data_root = DATASET_PATH
else:
    data_root = "/kaggle/input" if IS_KAGGLE else "/content"

if os.path.isfile(data_root) and data_root.endswith(".csv"):
    raw_csv_path = data_root
else:
    cands = sorted(
        glob.glob(f"{data_root}/**/metadata.csv", recursive=True)
        + glob.glob(f"{data_root}/**/*.csv", recursive=True)
    )
    print("CSV candidates:", cands)
    assert cands, f"Không tìm thấy file .csv trong {data_root}!"
    raw_csv_path = cands[0]

base_dir = os.path.dirname(raw_csv_path)
print("Raw CSV :", raw_csv_path)
print("Base dir:", base_dir)

def resolve_wav(wav_name: str):
    # Hỗ trợ trường hợp CSV ghi "utt_00000.wav" nhưng file thực tế nằm trong "wavs/utt_00000.wav" (hoặc ngược lại)
    for rel in (wav_name, os.path.join("wavs", wav_name), os.path.join("audio", wav_name), os.path.basename(wav_name)):
        full = os.path.join(base_dir, rel)
        if os.path.exists(full):
            return rel, full
    return None, None

good, bad, missing, filtered_dur = [], [], 0, 0
for i, line in enumerate(open(raw_csv_path, encoding="utf-8-sig"), 1):
    line = line.rstrip("\r\n")
    if not line.strip():
        continue
    parts = [p.strip() for p in line.split("|")]
    if i == 1 and parts[0].lower() in ("file", "filename", "path", "audio", "wav", "id"):
        continue
    if len(parts) >= 3:
        # Dạng: utt_00000.wav|ngochuyen|ma môn -> lấy cột đầu làm file, cột cuối làm text
        wav_raw, txt = parts[0], "|".join(parts[2:]).strip()
    elif len(parts) == 2:
        # Dạng: utt_00000.wav|ma môn
        wav_raw, txt = parts[0], parts[1]
    else:
        bad.append((i, line))
        continue

    if not wav_raw or not txt:
        bad.append((i, line))
        continue

    rel_wav, full_wav = resolve_wav(wav_raw)
    if full_wav is None:
        missing += 1
        continue

    try:
        dur = sf.info(full_wav).duration
    except Exception:
        bad.append((i, line))
        continue

    if not (MIN_DURATION <= dur <= MAX_DURATION):
        filtered_dur += 1
        continue

    # Ghi ra định dạng chuẩn 2 cột (file|text) cho single-speaker
    good.append((rel_wav, txt))

assert good, "Không có dòng dữ liệu hợp lệ nào sau khi lọc! Hãy kiểm tra lại đường dẫn audio_dir và CSV."
audio_dir = base_dir
csv_path = os.path.join(WORK, "metadata_single_clean.csv")
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("\n".join(f"{w}|{t}" for w, t in good) + "\n")

print(f"Kept: {len(good)} | Bad format: {len(bad)} | Missing wav: {missing} | Filtered duration (<{MIN_DURATION}s or >{MAX_DURATION}s): {filtered_dur}")
print("Sample rows:", good[:3])

In [ ]:
# 5. Tải base checkpoint single-speaker (Lessac medium 22050 Hz)
from huggingface_hub import hf_hub_download

BASE = hf_hub_download(
    BASE_REPO,
    BASE_CKPT_IN_REPO,
    repo_type="dataset",
    local_dir=f"{WORK}/base_ckpt",
)
print("Base checkpoint:", BASE)

In [ ]:
# 6. Huấn luyện (Single Speaker: --model.num_speakers 1)
# Tự động resume nếu đã có checkpoint trong OUT, hoặc bắt đầu từ BASE nếu chạy lần đầu
import glob
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

ckpts = sorted(
    glob.glob(f"{OUT}/lightning_logs/*/checkpoints/last.ckpt")
    + glob.glob(f"{OUT}/lightning_logs/*/checkpoints/*.ckpt"),
    key=os.path.getmtime,
)

if ckpts:
    RESUME_CKPT = ckpts[-1]
    print("Resume từ checkpoint:", RESUME_CKPT)
    ckpt_flag = f"--ckpt_path {RESUME_CKPT}"
else:
    print("Bắt đầu warmstart từ BASE:", BASE)
    ckpt_flag = f"--model.warmstart_ckpt {BASE}"

!python3 -m piper.train fit \
  --data.voice_name {VOICE} \
  --data.csv_path {csv_path} \
  --data.audio_dir {audio_dir} \
  --data.phoneme_type vietnamese \
  --data.espeak_voice vi \
  --data.cache_dir {WORK}/cache \
  --data.config_path {OUT}/{VOICE}.onnx.json \
  --data.batch_size {BATCH_SIZE} \
  --data.num_workers 4 \
  --data.length_bucketing true \
  --model.sample_rate 22050 \
  --model.num_speakers 1 \
  --model.keep_last_k 3 \
  --model.mos_metric none \
  {ckpt_flag} \
  --trainer.devices 1 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.default_root_dir {OUT}

In [ ]:
# 7. Xuất file ONNX
import glob
ckpt = sorted(glob.glob(f"{OUT}/lightning_logs/*/checkpoints/last.ckpt"))[-1]
print("Exporting checkpoint:", ckpt)
!python3 -m piper.train.export_onnx --checkpoint {ckpt} --output-file {OUT}/{VOICE}.onnx
!ls -lh {OUT}/{VOICE}.onnx {OUT}/{VOICE}.onnx.json

In [ ]:
# 8. Đọc thử file ONNX vừa xuất
from IPython.display import Audio, display

test_wav = os.path.join(WORK, "test_single.wav")
!echo "Chương hai nghìn tám trăm mười. Xin chào, đây là giọng đọc thử nghiệm của Ngọc Huyền." | python3 -m piper -m {OUT}/{VOICE}.onnx -f {test_wav}
display(Audio(test_wav))